In [1]:
import pandas as pd
from pathlib import Path

# ---------------- CONFIG ----------------
ESM_XLSX = Path("./carbon_density_BECCS_ESM_regional_paper.xlsx")
IAM_XLSX = Path("./carbon_density_BECCS_IAM_regional_paper.xlsx")

OUTPUT_XLSX = Path("scaling_factors_regions_biomass_ssp126_paper.xlsx")
OUTPUT_CSV  = Path("scaling_factors_regions_biomass_ssp126_paper.csv")

CANONICAL_SCENARIO = "SSP1-26" 

# Region mapping
REGION_MAP = {
    "Africa": "R10AFRICA",
    "Europe": "R10EUROPE",
    "Latin America and Caribbean": "R10LATIN_AM",
    "Middle East": "R10MIDDLE_EAST",
    "North America": "R10NORTH_AM",
    "Eastern Asia": "R10CHINA+",
    "Southern Asia": "R10INDIA+",
    "Asia-Pacific": "R10PAC_OECD",
    "Eurasia": "R10REF_ECON",
    "South-East Asia and developing Pacific": "R10REST_ASIA",
}

# Column names in ESM file
LSM_MODEL_COL = "Model"          # Land Surface Model (e.g., jules)
ESM_MODEL_COL = "ESM"            # GCM / ESM (e.g., mpi-esm1-2-hr)
ESM_SCENARIO_COL = "SSP"
ESM_REGION_COL = "Region"
ESM_TRANSITION_COL = "Landuse"
ESM_CARBON_COL = "Carbon_Density_tCO2_per_ha"

# IAM columns
IAM_SCENARIO_COL = "scenario"
IAM_REGION_COL = "Region"
IAM_CARBON_COL = "Carbon_Density_tCO2_per_ha"
IAM_MODEL_COL = None  # Set to actual column name if IAM has model splits you want to retain (e.g., "Model")

# If IAM has a model column and you want per-IAM model scaling, set this True
PER_IAM_MODEL = False

# Negative handling: 'keep', 'clip_zero', or 'drop'
NEGATIVE_POLICY = "keep"

# --------------- Helpers ---------------
def normalize_esm_scenario(v):
    return CANONICAL_SCENARIO if isinstance(v, str) and v.lower().strip() == "ssp126" else None

def normalize_iam_scenario(v):
    up = str(v).upper()
    return CANONICAL_SCENARIO if ("SSP1" in up and "26" in up) else None

# --------------- Load ---------------
print("Reading ESM:", ESM_XLSX)
esm = pd.read_excel(ESM_XLSX)
print("Reading IAM:", IAM_XLSX)
iam = pd.read_excel(IAM_XLSX)

# --------------- Scenario normalize ---------------
esm["SSP_std"] = esm[ESM_SCENARIO_COL].apply(normalize_esm_scenario)
iam["SSP_std"] = iam[IAM_SCENARIO_COL].apply(normalize_iam_scenario)

esm = esm[esm["SSP_std"] == CANONICAL_SCENARIO].copy()
iam = iam[iam["SSP_std"] == CANONICAL_SCENARIO].copy()

print("\nAfter scenario filter:")
print("  ESM rows:", len(esm))
print("  IAM rows:", len(iam))

# --------------- Region mapping ---------------
esm = esm[esm[ESM_REGION_COL] != "World"].copy()
esm["Region_mapped"] = esm[ESM_REGION_COL].map(REGION_MAP)

unmapped_regions = esm[esm["Region_mapped"].isna()][ESM_REGION_COL].unique()
if len(unmapped_regions) > 0:
    print("WARNING: Unmapped ESM regions (dropping):", unmapped_regions)
    esm = esm[esm["Region_mapped"].notna()].copy()

# --------------- Negative handling ---------------
neg_mask = esm[ESM_CARBON_COL] < 0
neg_count = neg_mask.sum()
if neg_count > 0:
    print(f"\nNOTE: {neg_count} ESM rows have negative carbon density (e.g., min={esm[ESM_CARBON_COL].min():.3f}). Policy: {NEGATIVE_POLICY}")
    if NEGATIVE_POLICY == "clip_zero":
        esm.loc[neg_mask, ESM_CARBON_COL] = 0.0
    elif NEGATIVE_POLICY == "drop":
        esm = esm.loc[~neg_mask].copy()

# --------------- Prepare IAM regions ---------------
iam = iam.rename(columns={IAM_REGION_COL: "Region_mapped"})

if IAM_MODEL_COL and PER_IAM_MODEL:
    # keep IAM model column
    pass
else:
    # aggregate across IAM model dimension if present (mean)
    group_keys = ["SSP_std", "Region_mapped"]
    iam = (iam
           .groupby(group_keys, as_index=False)
           .agg({IAM_CARBON_COL: "mean"}))

# --------------- ESM granularity aggregation ---------------
# If there are duplicate rows for same (SSP_std, ESM, LSM, Transition, Region_mapped), average them
esm_grouped = (esm
    .groupby(["SSP_std", ESM_MODEL_COL, LSM_MODEL_COL, ESM_TRANSITION_COL, "Region_mapped"], as_index=False)
    .agg({ESM_CARBON_COL: "mean"})
    .rename(columns={ESM_CARBON_COL: "ESM_CarbonDensity"}))

# --------------- IAM grouped ---------------
if IAM_MODEL_COL and PER_IAM_MODEL:
    iam_grouped = (iam
        .groupby(["SSP_std", IAM_MODEL_COL, "Region_mapped"], as_index=False)
        .agg({IAM_CARBON_COL: "mean"})
        .rename(columns={IAM_CARBON_COL: "IAM_CarbonDensity", IAM_MODEL_COL: "IAM_Model"}))
else:
    iam_grouped = iam.rename(columns={IAM_CARBON_COL: "IAM_CarbonDensity"})

print("\nESM grouped rows:", len(esm_grouped))
print("IAM grouped rows:", len(iam_grouped))

# --------------- Merge ---------------
if IAM_MODEL_COL and PER_IAM_MODEL:
    merged = esm_grouped.merge(iam_grouped, on=["SSP_std", "Region_mapped"], how="inner", validate="many_to_many")
else:
    merged = esm_grouped.merge(iam_grouped, on=["SSP_std", "Region_mapped"], how="inner", validate="many_to_one")

# --------------- Scaling ---------------
merged = merged[merged["IAM_CarbonDensity"] != 0].copy()
merged["scaling_factor"] = merged["ESM_CarbonDensity"] / merged["IAM_CarbonDensity"]

# Column ordering
cols = ["SSP_std", "Region_mapped", ESM_MODEL_COL, LSM_MODEL_COL, ESM_TRANSITION_COL,
        "ESM_CarbonDensity", "IAM_CarbonDensity", "scaling_factor"]
if "IAM_Model" in merged.columns:
    cols.insert(5, "IAM_Model")
merged = merged[cols].sort_values(["SSP_std", ESM_MODEL_COL, LSM_MODEL_COL, ESM_TRANSITION_COL, "Region_mapped"]).reset_index(drop=True)

print("\nPreview (first 15 rows):")
print(merged.head(15))

# Coverage diagnostics
expected_keys = esm_grouped[["SSP_std", ESM_MODEL_COL, LSM_MODEL_COL, ESM_TRANSITION_COL, "Region_mapped"]].drop_duplicates()
matched_keys = merged[["SSP_std", ESM_MODEL_COL, LSM_MODEL_COL, ESM_TRANSITION_COL, "Region_mapped"]].drop_duplicates()
missing = (expected_keys
           .merge(matched_keys, how="left", indicator=True)
           .query('_merge == \"left_only\"')
           .drop(columns="_merge"))
if not missing.empty:
    print("\nWARNING: Missing IAM matches for some ESM combinations:")
    print(missing.head(15))
    print("Total missing combinations:", len(missing))
else:
    print("\nAll ESM combinations matched IAM data (after filtering).")

# --------------- Save ---------------
OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
merged.to_excel(OUTPUT_XLSX, index=False)
merged.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved scaling factors to:\n  {OUTPUT_XLSX.resolve()}\n  {OUTPUT_CSV.resolve()}")

# --------------- Stats ---------------
print("\nScaling factor stats:")
print(merged["scaling_factor"].describe())

# Simple outlier flag (> 99th or < 1st percentile)
low_thr = merged["scaling_factor"].quantile(0.01)
high_thr = merged["scaling_factor"].quantile(0.99)
outliers = merged[(merged["scaling_factor"] < low_thr) | (merged["scaling_factor"] > high_thr)]
if not outliers.empty:
    print(f"\nPotential extreme outliers outside 1st–99th percentile ({len(outliers)} rows). Examples:")
    print(outliers.head(10))

Reading ESM: carbon_density_BECCS_ESM_regional_paper.xlsx
Reading IAM: carbon_density_BECCS_IAM_regional_paper.xlsx

After scenario filter:
  ESM rows: 198
  IAM rows: 10

ESM grouped rows: 180
IAM grouped rows: 10

Preview (first 15 rows):
    SSP_std   Region_mapped           ESM Model   Landuse  ESM_CarbonDensity  \
0   SSP1-26       R10AFRICA  ipsl-cm6a-lr   clm   agtobio           3214.136   
1   SSP1-26       R10CHINA+  ipsl-cm6a-lr   clm   agtobio           5406.828   
2   SSP1-26       R10EUROPE  ipsl-cm6a-lr   clm   agtobio           6246.327   
3   SSP1-26       R10INDIA+  ipsl-cm6a-lr   clm   agtobio           5869.383   
4   SSP1-26     R10LATIN_AM  ipsl-cm6a-lr   clm   agtobio           4088.615   
5   SSP1-26  R10MIDDLE_EAST  ipsl-cm6a-lr   clm   agtobio           3599.134   
6   SSP1-26     R10NORTH_AM  ipsl-cm6a-lr   clm   agtobio           5765.734   
7   SSP1-26     R10PAC_OECD  ipsl-cm6a-lr   clm   agtobio           3696.578   
8   SSP1-26     R10REF_ECON  ipsl-cm6a-